# Cartesian Series Extrapolation

This notebook downloads a SHARP CEA time sequence, trains the first observation as a regular single extrapolation, and then uses that first result as the `meta_path` start point for `nf2.run_series`.

Series runs need this initial extrapolation because each later time step starts from an already trained NF2 state instead of from random network weights. Edit the settings cell below, then run the notebook from top to bottom.

In [ ]:
from pathlib import Path

RUN_DOWNLOAD = True
RUN_TRAIN_INITIAL = True
RUN_TRAIN_SERIES = True

JSOC_EMAIL = "you@example.org"
SHARP_NUM = 377
NOAA_NUM = None
T_START = "2011-02-15T00:00:00"
T_END = "2011-02-15T01:00:00"
CADENCE = "720s"
SERIES = "sharp_cea_720s"
SEGMENTS = "Br,Bp,Bt,Br_err,Bp_err,Bt_err"

RUN_ROOT = Path("runs/cartesian_series")
DATA_DIR = RUN_ROOT / "data"
INITIAL_RUN_DIR = RUN_ROOT / "initial"
SERIES_RUN_DIR = RUN_ROOT / "series"
INITIAL_WORK_DIR = INITIAL_RUN_DIR / "work"
SERIES_WORK_DIR = SERIES_RUN_DIR / "work"
INITIAL_CKPT_PATH = INITIAL_RUN_DIR / "last.ckpt"
NF2_PATH = None
EXPORT_DIR = RUN_ROOT / "exports"

EXPORT_MM_PER_PIXEL = 1.44
HEIGHT_MIN = 0
HEIGHT_MAX = 80

In [ ]:
import os
os.environ.setdefault("WANDB_CONSOLE", "off")

from pathlib import Path
from dateutil.parser import parse

import matplotlib.pyplot as plt
import numpy as np
import wandb

import nf2
from nf2.evaluation.metric import normalized_divergence, sigma_J, theta_J, vector_norm
from nf2.evaluation.output_metrics import current_density

## 1. Download The SHARP CEA Sequence

Fill in `JSOC_EMAIL`, either `SHARP_NUM` or `NOAA_NUM`, and the time range in the settings cell. The download requests vector components and their uncertainty maps. The helper below matches exact segment suffixes such as `.Br.fits` and `.Br_err.fits`, so field files and error files do not get mixed.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
if RUN_DOWNLOAD:
    nf2.download_sharp_series(
        str(DATA_DIR), JSOC_EMAIL, parse(T_START), parse(T_END),
        noaa_num=NOAA_NUM, sharp_num=SHARP_NUM, cadence=CADENCE,
        segments=SEGMENTS, series=SERIES,
    )

def segment_files(segment):
    return sorted(DATA_DIR.glob(f"*.{segment}.fits"))

series_files = {segment: segment_files(segment) for segment in ["Br", "Bt", "Bp", "Br_err", "Bt_err", "Bp_err"]}
n_steps = len(series_files["Br"])
lengths = {segment: len(files) for segment, files in series_files.items()}
if n_steps == 0 or any(length != n_steps for length in lengths.values()):
    raise ValueError(f"Downloaded files do not form a complete series: {lengths}")

first_files = {segment: files[0] for segment, files in series_files.items()}
print(f"Found {n_steps} complete SHARP CEA time steps.")
print("First observation used for the initial meta-path run:")
for segment in ["Br", "Bt", "Bp", "Br_err", "Bt_err", "Bp_err"]:
    print(segment, first_files[segment])

## 2. Train The Initial Observation

The first observation is trained as a normal Cartesian extrapolation when `RUN_TRAIN_INITIAL = True`. Its `last.ckpt` file, `INITIAL_CKPT_PATH`, becomes the `meta_path` for the series run. Keep this file available until the series has started successfully. The notebook uses zero DataLoader workers so the example remains stable in interactive runtimes.

Adjust `HEIGHT_MIN`, `HEIGHT_MAX`, batch sizes, losses, or training settings here if you want the first state and the series states to use different training choices.

In [ ]:
initial_losses = [
    {"type": "boundary", "name": "boundary", "weight": 1.0, "datasets": "boundary"},
    {"type": "boundary", "name": "potential_boundary", "weight": 10.0, "datasets": "potential"},
    {"type": "force_free", "name": "force_free", "weight": 1.0e-4, "datasets": ["random"]},
    {"type": "potential", "name": "potential", "weight": {"type": "step", "steps": 5000, "start": 1.0e-4, "end": 0.0}, "datasets": ["random"]},
]

initial_config = {
    "path": str(INITIAL_RUN_DIR),
    "work_path": str(INITIAL_WORK_DIR),
    "logging": {"project": "nf2", "name": "SHARP CEA series initial"},
    "data": {
        "geometry": "cartesian",
        "boundaries": [{
            "id": "boundary",
            "type": "sharp",
            "files": {"Br": str(first_files["Br"]), "Bt": str(first_files["Bt"]), "Bp": str(first_files["Bp"])},
            "errors": {"Br_err": str(first_files["Br_err"]), "Bt_err": str(first_files["Bt_err"]), "Bp_err": str(first_files["Bp_err"])},
        }],
        "potential_boundary": {"type": "potential", "strides": 4},
        "validation": [
            {"id": "boundary", "type": "sharp", "files": {"Br": str(first_files["Br"]), "Bt": str(first_files["Bt"]), "Bp": str(first_files["Bp"])}, "errors": {"Br_err": str(first_files["Br_err"]), "Bt_err": str(first_files["Bt_err"]), "Bp_err": str(first_files["Bp_err"])}},
            {"id": "slices", "type": "slices"},
        ],
        "z_range": [HEIGHT_MIN, HEIGHT_MAX],
        "num_workers": 0,
        "train_num_workers": 0,
        "validation_num_workers": 0,
    },
    "losses": initial_losses,
    "loss_scaling": [{"type": "b_height", "name": "b_height", "loss_ids": ["force_free", "potential"]}],
    "callbacks": [{"type": "boundary", "dataset": "boundary"}, {"type": "slices", "dataset": "slices"}],
}

if RUN_TRAIN_INITIAL:
    nf2.run(**initial_config)
    wandb.finish()
else:
    initial_config

## 3. Train The Full Series

`nf2.run_series` runs when `RUN_TRAIN_SERIES = True` and receives the completed first extrapolation through `meta_path`. The `data_path` entries below are glob patterns for all downloaded time steps; each component must match the same number of files and sort in chronological order. The notebook uses explicit patterns for field and uncertainty files so the series loader pairs `Br`, `Bt`, `Bp`, `Br_err`, `Bt_err`, and `Bp_err` consistently. Series data modules are loaded lazily in the notebook to avoid recursive worker/preload failures in interactive runtimes; `reload=True` rebuilds any cached data-module state from older notebook runs.

In [ ]:
series_losses = [
    {"type": "boundary", "name": "boundary", "weight": 1.0, "datasets": "boundary"},
    {"type": "boundary", "name": "potential_boundary", "weight": 10.0, "datasets": "potential"},
    {"type": "force_free", "name": "force_free", "weight": 1.0e-4, "datasets": ["random"]},
    {"type": "potential", "name": "potential", "weight": 0.0, "datasets": ["random"]},
]

series_config = {
    "path": str(SERIES_RUN_DIR),
    "work_path": str(SERIES_WORK_DIR),
    "meta_path": str(INITIAL_CKPT_PATH),
    "logging": {"project": "nf2", "name": "SHARP CEA series"},
    "data": {
        "geometry": "cartesian",
        "boundaries": [{
            "id": "boundary",
            "type": "fits",
            "data_path": {
                "Br": str(DATA_DIR / "*.Br.fits"),
                "Bt": str(DATA_DIR / "*.Bt.fits"),
                "Bp": str(DATA_DIR / "*.Bp.fits"),
                "Br_err": str(DATA_DIR / "*.Br_err.fits"),
                "Bt_err": str(DATA_DIR / "*.Bt_err.fits"),
                "Bp_err": str(DATA_DIR / "*.Bp_err.fits"),
            },
        }],
        "potential_boundary": {"type": "potential", "strides": 4},
        "z_range": [HEIGHT_MIN, HEIGHT_MAX],
        "iterations": 2000,
        "num_workers": 0,
        "train_num_workers": 0,
        "validation_num_workers": 0,
        "data_module_workers": 1,
        "preload_data_modules": False,
    },
    "training": {"reload_dataloaders_every_n_epochs": 1, "check_val_every_n_epoch": 10},
    "losses": series_losses,
    "loss_scaling": [{"type": "b_height", "name": "b_height", "loss_ids": ["force_free", "potential"]}],
    "callbacks": [{"type": "boundary", "dataset": "boundary"}, {"type": "metrics", "dataset": "cube"}, {"type": "slices", "dataset": "slices"}],
}

if RUN_TRAIN_SERIES:
    if not INITIAL_CKPT_PATH.exists():
        raise FileNotFoundError(f"Initial meta-path checkpoint is missing: {INITIAL_CKPT_PATH}. Run the previous cell first.")
    nf2.run_series(**series_config, reload=True)
    wandb.finish()
else:
    series_config

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
nf2.export_series([str(SERIES_RUN_DIR / "*.nf2")], str(EXPORT_DIR), fmt="hdf5", overwrite=True, Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "alpha", "free_energy"])

nf2_files = sorted(SERIES_RUN_DIR.glob("*.nf2"))
NF2_PATH = Path(NF2_PATH) if NF2_PATH else (nf2_files[0] if nf2_files else None)
if NF2_PATH is None:
    raise FileNotFoundError("No .nf2 series state found yet.")

In [ ]:
out = nf2.load(NF2_PATH)
cube = out.load_cube(
    Mm_per_pixel=EXPORT_MM_PER_PIXEL,
    height_range=[HEIGHT_MIN, HEIGHT_MAX],
    metrics=["j", "j_vec", "free_energy"],
    progress=True,
)

def values(x):
    return np.asarray(getattr(x, "value", x))

b = values(cube["b"])
j = values(cube["metrics"]["j"])
j_vec = values(cube["metrics"]["j_vec"])
free_energy = values(cube["metrics"]["free_energy"])
metrics = {
    "mean_normalized_divergence": float(np.nanmean(normalized_divergence(b))),
    "theta_J_deg": float(theta_J(b, j_vec)),
    "sigma_J_1e2": float(sigma_J(b, j_vec) * 1e2),
    "mean_B_norm": float(np.nanmean(vector_norm(b))),
    "mean_J_norm": float(np.nanmean(j)),
    "total_free_energy_density": float(np.nansum(free_energy)),
}
metrics

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

current_map = np.nansum(j, axis=2)
free_energy_map = np.nansum(free_energy, axis=2)
boundary_bz = b[:, :, 0, 2]

im = axs[0].imshow(current_map.T, origin="lower", cmap="magma")
axs[0].set_title("Integrated |J|")
plt.colorbar(im, ax=axs[0], fraction=0.046)

im = axs[1].imshow(free_energy_map.T, origin="lower", cmap="inferno")
axs[1].set_title("Free energy")
plt.colorbar(im, ax=axs[1], fraction=0.046)

lim = np.nanpercentile(np.abs(boundary_bz), 99)
im = axs[2].imshow(boundary_bz.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
axs[2].set_title("Model boundary Bz")
plt.colorbar(im, ax=axs[2], fraction=0.046)

for ax in axs:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()